## ETL of GeoSpatial data from US Census to Hugging Face

The US Census has restrictions on how it serves files.  So we'll copy them elsewhere first.


## Setup


In [ ]:
%%capture
%%bash
apt-get update
apt-get install -y tree jq


In [ ]:
import os
import pandas as pd
from google.colab import userdata


## Get a list of all shapefiles


In [ ]:
url = 'https://www2.census.gov/geo/tiger/TIGER2024/ROADS/'
url


In [ ]:
tables = pd.read_html( url )
len(tables)


In [ ]:
df = tables[0]
df


In [ ]:
shapefiles = [ x for x in df["Name"].dropna() if x.endswith("zip") ]
shapefiles[:5]


In [ ]:
len(shapefiles)


In [ ]:
with open("shapefiles", "w") as f:
  f.writelines(f"{url + x}\n" for x in shapefiles )


In [ ]:
! ls -l

In [ ]:
! head -3 shapefiles


In [ ]:
! wc -l shapefiles


## Download shapefiles




### States


In [ ]:
states_url = "https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip"
states_url


In [ ]:
# Shapefiles for states
! mkdir -p TIGER2024/STATE/ ; cd TIGER2024/STATE/ ; curl -L -O {states_url}


### Roads


In [ ]:
# Shapefiles for roads
%%bash

cut -c3- << 'eof' | tee /tmp/tiger.roads.sh

  mkdir -p TIGER2024/ROADS/
  cd TIGER2024/ROADS/
  time -p cat ../../shapefiles |
    xargs -n 1 -P 40 -t curl -s -O -L
  echo == done
  rm -f ${0}
  sleep 1000

eof

tmux new-session -s shapefiles -d 'bash /tmp/tiger.roads.sh'


In [ ]:
# Wait for roads downloads to finish
%%capture
%%bash

while [ -f /tmp/tiger.roads.sh ] ; do
  date
  sleep 1
done


### Verify


In [ ]:
! tree TIGER2024/


In [ ]:
! find ./TIGER2024/ | grep -c zip$


## Push to Hugging Face


## Configure settings and authenticate to Hugging Face


In [ ]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account


In [ ]:
hf_repo = os.environ["HF_REPO"] = "DUC-2026"
hf_repo


In [ ]:
hf_file = os.environ["HF_FILE"] = "TIGER2024/"
hf_file


In [ ]:
# Retrieve the token from Colab's secret manager
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"


In [ ]:
# Log in automatically without an interactive prompt
! hf auth login --token ${HF_TOKEN}


## Push the parquet file to Hugging Face


In [ ]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [ ]:
print(hf_upload.stdout)


In [ ]:
%%capture hf_repo_list
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
echo DRY-RUN: \
hf datasets  \
  list -R \
  "${TARGET_REPO}"/ \


In [ ]:
print(hf_repo_list.stdout)
